### ЗАДАЧА: Пакетная загрузка накладных (try/except + custom exceptions)

Из внешней системы приходят строки с накладными.
Нужно безопасно разобрать их, отделить валидные записи от ошибок и собрать краткий отчёт.

НЕОБХОДИМО РЕАЛИЗОВАТЬ:

1. Иерархию кастомных исключений:
   - `InvoiceError`
   - `RowFormatError`
   - `QuantityError`
   - `PriceError`
   - `StatusError`.

2. Функцию `parse_invoice(row)`:
   - формат: `invoice_id,item,quantity,price,status`
   - `quantity` должен быть целым числом и `> 0`
   - `price` должен быть числом и `> 0`
   - допустимые статусы: `new`, `approved`, `paid`
   - при ошибке конвертации использовать `raise ... from ...`.

3. Функцию `load_invoices(rows)`:
   - вернуть `(invoices, errors)`
   - ошибки хранить как `(row, error_type, message)`
   - не останавливать цикл на первой ошибке.

4. Вывести:
   - число валидных накладных,
   - ошибки по типам,
   - сумму только для накладных со статусом `paid`,
   - товар-лидер по суммарному количеству в валидных накладных.

ПОДСКАЗКИ:
- `quantity` и `price` сначала конвертируются, потом валидируются.
- Для ошибок по типам удобно собирать обычный словарь-счётчик.


In [24]:
rows = [
    'INV-100,Keyboard,3,120,paid',
    'INV-101,Mouse,0,40,new',
    'INV-102,Monitor,2,abc,approved',
    'INV-103,Laptop,1,1400,shipped',
    'INV-104,Keyboard,5,110,paid',
    'INV-105,Dock,2,-50,approved',
]


class InvoiceError(Exception):
    pass


class RowFormatError(InvoiceError):
    pass


class QuantityError(InvoiceError):
    pass


class PriceError(InvoiceError):
    pass


class StatusError(InvoiceError):
    pass


def parse_invoice(row):
    # TODO: распарсить строку и провалидировать quantity, price, status
    if len(row.split(",")) != 5:
        raise RowFormatError("В строке не 5 элементов")
    
    id, item, quantity, price, status = row.split(",")
    try:
        quantity = int(quantity)
    except ValueError as e:
        raise QuantityError("Не является числом") from e
    
    if quantity < 0:
        raise QuantityError("Количество не может быть отрицательным")

    try:
        price = int(price)
    except ValueError as e:
        raise PriceError("Цена должна быть числом") from e
    
    if price < 0:
        raise PriceError("Цена не может быть отрицательной")
    
    allowed_statuses = {"paid", "new", "approved"}
    if status not in allowed_statuses:
        raise StatusError("Недопустимый статус")

    return {
        "id": id,
        "item": item,
        "quantity": quantity,
        "price": price,
        "status": status
    }



def load_invoices(rows):
    # TODO: вернуть (invoices, errors)
    invoices = []
    errors = []
    for row in rows:
        try:
            invoices.append(parse_invoice(row))
        except InvoiceError as e:
            errors.append((row, type(e).__name__, e))
    return invoices, errors

invoices, errors = load_invoices(rows)
print(f"Количество валидных накладных = {len(invoices)} шт.")
print(f"Количество ошибочных накладных = {len(errors)} шт.")
type_errors = {}
for row, error, name in errors:
    print (row, error, name)
    if error not in type_errors:
        type_errors[error] = type_errors.get(error, 0)
        type_errors[error] += 1
print(type_errors)
        



# TODO: вызвать load_invoices(rows)
# TODO: вывести число валидных накладных и число ошибок
# TODO: вывести ошибки по типам
# TODO: посчитать paid_total
# TODO: найти товар-лидер по количеству


Количество валидных накладных = 3 шт.
Количество ошибочных накладных = 3 шт.
INV-102,Monitor,2,abc,approved PriceError Цена должна быть числом
INV-103,Laptop,1,1400,shipped StatusError Недопустимый статус
INV-105,Dock,2,-50,approved PriceError Цена не может быть отрицательной
{'PriceError': 1, 'StatusError': 1}
